# KAT-VAD v5 R7 — Phase 1: CLIP **14×14×512** patch grids (Nexar train/val + every DoTA-CAP clip)

Rules: `core/docs/v5/PREREG_V5.md` **§4** (R7, DRAFT — Phase 1 extracts features only and reads no number; §4 is
signed before any R7 training or read). Plan `.project/plans/katvad-v5-r7-clip.md`. Setup `core/docs/v5/R7_CLIP_SETUP.md`.

| step | what | GPU? |
|---|---|---|
| 1 | CLIP ViT-B/16 patch grids, **unpooled** `(rows, 14, 14, 512)` fp16: Nexar train **s8**, Nexar val **s3** (`v5_stage1 extract --clip-grid`), **all 1,129** DoTA-CAP clips **s3** (`v5_stage0 extract --clip-grid`, the 5 CAP groups in order) — three jobs in parallel | yes |
| 2 | gate summary → paste `REPORTS/r7_phase1.md` to Claude | no |

**Gates (hard, a failing video / clip stores nothing).** Nexar G1: decoded length == `Nexar_s1_ncc`; max |projected
CLS − `image_embeds`| ≤ 1e-5; every stored row's `image_embeds` vs `Nexar_s1_ncc` cos ≥ 0.999. DoTA-CAP G1 as Stage 0
(projected CLS; every row vs `DoTA_s1_ncc` ≥ 0.95, Amendment 2) and **G-R7c** on CAP-dev: the lag-1 change map of the
stored fp16 grid equals Stage 0's `dc14` (max |Δ| ≤ 5e-3). **Pass = every Nexar video and every one of the 1,129 clips
stored, and all 569 CAP-dev clips compared with Stage 0.**

**What is stored (and what is not).** The 512-d tokens are `visual_projection(post_layernorm(patch))`, the space of
`image_embeds` and of check C's C14. Any pooling (7×7, 2×2, …) is a reshape-mean of these files later, never a
re-extraction. The 768-d hidden states before the projection are **not** stored. Rows are the v5.1 rows (Nexar s8 / s3,
DoTA-CAP s3); another stride needs a new extraction (Nexar s1 would be ~182 GB).

**Drive.** Code `Thesis-V5/kat-vad` (upload branch `v5-r7`). Writes only `Thesis-V5/cache/video_grid/clip_vit_b16/`
(~41 GB: train 16.4, val 17.2, DoTA-CAP 7.7) and `Thesis-V5/outputs/v5/v5_r7/REPORTS/`. `Thesis/`, `Thesis-V2/`
read-only. Resumable: a re-run skips finished videos; finished caches whose gate PASSed are not touched.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar, DoTA, MMAU/CAP), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: v2 caches / reports, DoTA-CAP alignment, Stage 0 maps
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # code + everything R7 writes
os.environ['PROJECT_ROOT']       = DRIVE_V5
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V5}/outputs'

os.environ['REPO'] = f'{DRIVE_V5}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5-r7 branch there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_CLIP_GRID_TAG'), 'this upload predates R7 -- re-upload kat-vad/ from v5-r7'

MAX_GPU_JOBS = 3      # concurrent GPU processes; 1 = sequential (set 1 on CUDA OOM, finished work is skipped)
MIRROR_EVERY = 300    # seconds between grid mirrors VM -> Drive
BATCH        = 128    # CLIP frames per forward

DATASET  = constants.NEXAR_DATASET
C        = constants.NEXAR_CONSTRUCTION_WINDOW
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
TAG      = constants.V5_CLIP_GRID_TAG                             # cgrid14
S_TRAIN, S_VAL, S_DOTA = (constants.V5_R7_NEXAR_TRAIN_STRIDE, constants.V5_R7_NEXAR_VAL_STRIDE,
                          constants.V5_R7_DOTA_STRIDE)            # 8, 3, 3

V2_CACHE = Path(DRIVE_V2) / 'cache'
V2_OUT   = Path(DRIVE_V2) / 'outputs'
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET    # Nexar G1 reference
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'                # DoTA-CAP G1 reference
CAP_RAW  = constants.DATA_ROOT / 'MMAU' / 'raw'
DCAP     = V2_OUT / 'REPORTS' / 'dota_cap'
ALIGN, FINAL = DCAP / 'dota_cap_alignment.json', DCAP / 'dota_cap_readout.json'
CAP_GROUPS = ['1-10', '11', '12-42', '43', '44-62']
S0_MAPS = next((Path(r) / 'outputs' / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME / 'maps'
                for r in (DRIVE_V5, DRIVE_V2)
                if (Path(r) / 'outputs' / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME / 'maps').is_dir()),
               None)                                              # G-R7c reference (Stage 0 dc14)

GRIDS    = Path(DRIVE_V5) / 'cache' / constants.V5_GRID_CACHE_DIRNAME / constants.V5_CLIP_GRID_ENCODER
D_GRID   = {'train': GRIDS / f'{DATASET}_s{S_TRAIN}_{TAG}', 'val': GRIDS / f'{DATASET}_s{S_VAL}_{TAG}',
            'dota': GRIDS / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_{TAG}'}
ROOT     = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / 'v5_r7'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/v5r7')
L_GRID   = {k: LOCAL / 'grids' / v.name for k, v in D_GRID.items()}
L_REP    = LOCAL / 'reports'          # every tool writes here; mirrored to REPORTS
LOGS     = LOCAL / 'logs'
for p in (REPORTS, L_REP, LOGS, *D_GRID.values(), *L_GRID.values()):
    p.mkdir(parents=True, exist_ok=True)
assert constants.OUTPUT_ROOT == Path(DRIVE_V5) / 'outputs', constants.OUTPUT_ROOT

for name, path in (('Nexar raw', RAW), ('census', CENSUS), ('Nexar_window', DATA), ('Nexar clip s1', CLIP_S1),
                   ('DoTA clip s1', DOTA_S1), ('CAP chunks', CAP_RAW / 'CAP-DATA_chunks'),
                   ('CAP annotation', CAP_RAW / 'cap_text_annotations.xls'), ('alignment', ALIGN),
                   ('final ids', FINAL), ('Stage 0 maps', S0_MAPS)):
    assert path is not None and path.exists(), f'missing {name}: {path}'
    print(f'  {name:16s} {path}   OK')
n_maps = len(list(S0_MAPS.glob('*.npz')))
assert n_maps == 569, f'Stage 0 maps: {n_maps}, expected 569 (CAP-dev)'
print('grids ->', GRIDS, '\nREPORTS ->', REPORTS)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd pandas
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 1 needs one'
nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
CPUS = os.cpu_count() or 8
PART = constants.CACHE_PART_SUFFIX
assert shutil.disk_usage('/content').free / 1e9 >= 90, 'need >= 90 GB free on /content (CLIP grids ~41 GB + CAP stream)'


def run(cmd, log=None):
    '''One CPU / sequential command, output streamed here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def last_line(path):
    try:
        lines = [l for l in path.read_text(errors='replace').splitlines() if l.strip()]
    except FileNotFoundError:
        return '(no output yet)'
    return lines[-1][-160:] if lines else '(no output yet)'


class Job:
    '''Commands run in order in one process slot; ``after`` runs in this kernel once all succeeded.'''

    def __init__(self, name, cmds, after=None):
        self.name, self.after = name, after
        self.cmds = [[str(c) for c in cmd] for cmd in cmds]
        self.log = LOGS / f'{name}.log'


def run_parallel(jobs, max_jobs=MAX_GPU_JOBS, every=120):
    '''Run independent jobs ``max_jobs`` at a time, one GPU. A failure does not stop the others; the cell raises
    after all have ended, with each failed job's log tail.'''
    if not jobs:
        print('nothing to run')
        return
    slots = max(1, min(max_jobs, len(jobs)))
    env = dict(ENV, OMP_NUM_THREADS=str(max(1, CPUS // slots)), MKL_NUM_THREADS=str(max(1, CPUS // slots)))
    pending, running, failed, t0, shown = list(jobs), {}, [], time.time(), time.time()

    def launch(job, i):
        fh = open(job.log, 'a')
        fh.write(f'\n$ {" ".join(job.cmds[i])}\n')
        fh.flush()
        return subprocess.Popen(job.cmds[i], cwd=str(REPO), env=env, stdout=fh, stderr=subprocess.STDOUT), fh

    def stamp():
        return f'[{(time.time() - t0) / 60:6.1f} min]'

    while pending or running:
        while pending and len(running) < slots:
            job = pending.pop(0)
            job.t0 = time.time()
            running[job.name] = (job, 0, *launch(job, 0))
            print(f'{stamp()} start {job.name}  (log {job.log})')
        time.sleep(10)
        for name, (job, i, proc, fh) in list(running.items()):
            rc = proc.poll()
            if rc is None:
                continue
            fh.close()
            del running[name]
            if rc:
                failed.append(job)
                print(f'{stamp()} FAILED {name} (exit {rc})')
            elif i + 1 < len(job.cmds):
                running[name] = (job, i + 1, *launch(job, i + 1))
            else:
                try:
                    if job.after:
                        job.after()
                    print(f'{stamp()} done {name} in {(time.time() - job.t0) / 60:.1f} min')
                except Exception as exc:  # a failed publish / check is this job's failure, not the others'
                    failed.append(job)
                    print(f'{stamp()} FAILED {name} after the run: {exc!r}')
        if running and time.time() - shown > every:
            shown = time.time()
            for name, (job, *_rest) in running.items():
                print(f'           {name}: {last_line(job.log)}')
    if failed:
        for job in failed:
            tail = job.log.read_text(errors='replace').splitlines()[-30:]
            print('=' * 30, f'\n{job.name} -- paste to Claude:\n' + '\n'.join(tail))
        raise RuntimeError(f'failed: {[j.name for j in failed]}')


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file() and not p.name.endswith(PART)]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def sync_dir(src, dst, restore=False):
    '''Copy every finished file of ``src`` missing (or, for json / npz, changed) in ``dst``; in-flight writes skipped.'''
    n = 0
    for f in sorted(src.glob('*')):
        if not f.is_file() or f.name.endswith(PART) or f.name.endswith('.partial'):
            continue
        g = dst / f.name
        small = f.suffix in ('.json', '.npz', '.md')
        if g.exists() and (g.stat().st_size == f.stat().st_size) and not (small and not restore):
            continue
        if restore:
            dst.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, g)
        else:
            publish(f, g)
        n += 1
    return n


class Mirror(threading.Thread):
    '''Step 1: copy finished grid files + reports VM -> Drive every ``MIRROR_EVERY`` s, and once more at stop.'''

    def __init__(self, pairs):
        super().__init__(daemon=True)
        self.pairs, self.halt, self.copied = pairs, threading.Event(), 0

    def run(self):
        while not self.halt.wait(MIRROR_EVERY):
            self.sync()

    def sync(self):
        for src, dst in self.pairs:
            try:
                self.copied += sync_dir(src, dst)
            except OSError as exc:     # a Drive hiccup: the next round retries
                print('mirror:', exc)

    def stop(self):
        self.halt.set()
        self.join()
        self.sync()
        print(f'mirror: {self.copied} files copied to Drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def show(name):
    p = L_REP / f'{name}.md'
    print(p.read_text() if p.exists() else f'(no {p.name})')


sync_dir(REPORTS, L_REP, restore=True)     # earlier sessions' reports: every skip below reads them locally
print('reports restored:', sorted(p.name for p in L_REP.glob('*.json')))

## 1. CLIP 14×14×512 grids — three caches in parallel (GPU)

* **Nexar train** positives at **s8**, **Nexar val** at **s3** (`v5_stage1 extract --clip-grid`), G1 vs `Nexar_s1_ncc`.
* **DoTA-CAP**, all 1,129 clips at **s3**, the 5 CAP groups one after the other in one job (`v5_stage0 extract
  --clip-grid`), G1 vs `DoTA_s1_ncc` + G-R7c vs Stage 0's `dc14` on CAP-dev.

Grids are written on VM disk and mirrored to Drive every `MIRROR_EVERY` s (atomic copies), so a killed session resumes.

In [ ]:
from core.data.v2_splits import load_split
from core.data.windows import load_windows

census = json.loads(CENSUS.read_text())
TRAIN_IDS = sorted({w.source for w in load_windows(DATA).values()})
VAL_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_VAL))
CAP_IDS = sorted(load_split(CAP_DEV))
assert all(census['videos'][v]['label'] == 1 for v in TRAIN_IDS), 'a window source is not a positive'
L_S0 = stage(S0_MAPS, LOCAL / 'stage0_maps')          # 569 small npz: read locally by the DoTA job


def nexar_gate(stride, ids):
    p = L_REP / f'g1_extract_cgrid_s{stride}.json'
    if not p.exists():
        return False
    r = json.loads(p.read_text())
    return r['pass'] and r['videos'] == len(ids) and r['ok'] == len(ids)


def dota_summary():
    '''Combine the 5 group reports. Pass iff all 1,129 clips stored and every CAP-dev clip compared with Stage 0.'''
    report = {}
    for g in CAP_GROUPS:
        p = L_REP / f'extract_cgrid_dota_{g}.json'
        if p.exists():
            report |= json.loads(p.read_text())
    ok = {v for v, e in report.items() if e.get('reason') == 'ok'}
    failed = {v: e.get('reason') for v, e in report.items() if e.get('reason') != 'ok'}
    gaps = {v: report[v].get('dc14_max_abs') for v in CAP_IDS if v in ok}
    uncompared = sorted(v for v in CAP_IDS if gaps.get(v) is None)
    cos = [e['pixel_min_cos'] for e in report.values() if e.get('pixel_min_cos') is not None]
    s = {'rule': 'PREREG_V5 §4 R7 Phase 1: DoTA-CAP G1 + G-R7c (CLIP 14x14x512, s3)', 'clips_reported': len(report),
         'ok': len(ok), 'rows': sum(report[v].get('steps', 0) for v in ok), 'failed': failed,
         'cap_dev_uncompared': uncompared, 'g1_pixel_min_cos': min(cos, default=None),
         'dc14_max_abs': max((g for g in gaps.values() if g is not None), default=None),
         'dc14_bar': constants.V5_R7_DC14_MAX_ABS,
         'pass': len(ok) == 1129 and not failed and not uncompared}
    (L_REP / 'g_r7_dota_cgrid.json').write_text(json.dumps(s, indent=2))
    (L_REP / 'g_r7_dota_cgrid.md').write_text(
        f"# R7 DoTA-CAP CLIP grids: **{'PASS' if s['pass'] else 'FAIL'}**\n\n{s['ok']} / {len(report)} clips, "
        f"{s['rows']} rows stored; failed {len(failed)} {sorted(failed.items())[:10]}; CAP-dev not compared with "
        f"Stage 0 {len(uncompared)} {uncompared[:10]}\n\nG1 pixel min cos {s['g1_pixel_min_cos']}; G-R7c max |Δ dc14| "
        f"{s['dc14_max_abs']} (bar {constants.V5_R7_DC14_MAX_ABS})\n")
    return s


def dota_done():
    p = L_REP / 'g_r7_dota_cgrid.json'
    return p.exists() and json.loads(p.read_text())['pass']


def nexar_job(tag, stride, ids):
    return Job(f'extract_{tag}', [[
        sys.executable, '-m', 'core.tools.v5_stage1', 'extract', '--clip-grid', '--root', RAW, '--census', CENSUS,
        '--ids-file', write_ids(LOCAL / f'ids_{tag}.txt', ids), '--clip-s1-dir', CLIP_S1,
        '--grid-dir', L_GRID[tag], '--stride', stride, '--report', L_REP / f'extract_cgrid_{tag}.json',
        '--batch-size', BATCH, '--device', 'auto', '--out-dir', L_REP]])


def dota_job():
    import pandas as pd
    ann = pd.read_excel(CAP_RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
    type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
    alignment = json.loads(ALIGN.read_text())
    final_ids = set(json.loads(FINAL.read_text())['ids'])
    need_cap = {alignment['clips'][d]['cap_id'] for d in final_ids}
    assert len(final_ids) == 1129, len(final_ids)
    cmds = []
    for g in CAP_GROUPS:
        lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
        ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
        parts = sorted((CAP_RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
        assert parts, f'no parts for group {g}'
        if ids:
            cmds.append([sys.executable, '-m', 'core.tools.v5_stage0', 'extract', '--clip-grid',
                         '--alignment', ALIGN, '--final-readout', FINAL, '--parts', *parts,
                         '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1,
                         '--stage0-maps-dir', L_S0,
                         '--report', L_REP / f'extract_cgrid_dota_{g}.json', '--out-dir', LOCAL / 's0out',
                         '--grid-dir', L_GRID['dota'], '--cap-ids-file', write_ids(LOCAL / f'cap_ids_{g}.txt', ids),
                         '--batch-size', BATCH, '--max-batch-gb', str(constants.MMAU_STREAM_BATCH_GB),
                         '--device', 'auto'])
    return Job('extract_dota', cmds)


todo = {'train': not nexar_gate(S_TRAIN, TRAIN_IDS), 'val': not nexar_gate(S_VAL, VAL_IDS), 'dota': not dota_done()}
print('to extract:', [k for k, v in todo.items() if v] or 'nothing')
for k, need in todo.items():
    if need:                          # resume: Drive's partial copy back to VM disk, the tools skip finished videos
        print(f'{k}: restored {sync_dir(D_GRID[k], L_GRID[k], restore=True)} files from Drive')
jobs = ([nexar_job('train', S_TRAIN, TRAIN_IDS)] if todo['train'] else []) + \
       ([nexar_job('val', S_VAL, VAL_IDS)] if todo['val'] else []) + ([dota_job()] if todo['dota'] else [])
mirror = Mirror([(L_GRID[k], D_GRID[k]) for k, v in todo.items() if v] + [(L_REP, REPORTS)])
mirror.start()
t0 = time.time()
try:
    run_parallel(jobs)
finally:
    mirror.stop()
print(f'step 1 wall {(time.time() - t0) / 60:.0f} min')
if jobs:
    flush_drive()

## 2. Gate summary → **paste `REPORTS/r7_phase1.md` to Claude**

Nothing here reads a detection or map number: Phase 1 only stores features. If any gate FAILs, stop and paste the
file; do not delete grids by hand.

In [ ]:
nexar = {k: json.loads((L_REP / f'g1_extract_cgrid_s{s}.json').read_text()) for k, s in (('train', S_TRAIN), ('val', S_VAL))}
dota = dota_summary()
sizes = {k: sum(f.stat().st_size for f in D_GRID[k].glob('*.npy')) / 1e9 for k in D_GRID}
ok = all(r['pass'] for r in nexar.values()) and dota['pass']
lines = [f"# R7 Phase 1 -- CLIP 14x14x512 grids: **{'PASS' if ok else 'FAIL'}**", '',
         'Rule: PREREG_V5 §4 (R7 Phase 1, features only; no number read).', '',
         '| cache | videos / clips ok | rows | G1 min cos | max abs CLS | G-R7c max abs dc14 | GB on Drive | gate |',
         '|---|---|---|---|---|---|---|---|']
for k, r in nexar.items():
    lines.append(f"| Nexar {k} s{r['stride']} | {r['ok']} / {r['videos']} | {r['rows']} | {r['g1_min_cos']} | "
                 f"{r['g1_max_abs']} | -- | {sizes[k]:.1f} | {'PASS' if r['pass'] else 'FAIL'} |")
lines.append(f"| DoTA-CAP s{S_DOTA} | {dota['ok']} / {dota['clips_reported']} | {dota['rows']} | "
             f"{dota['g1_pixel_min_cos']} | -- | {dota['dc14_max_abs']} | {sizes['dota']:.1f} | "
             f"{'PASS' if dota['pass'] else 'FAIL'} |")
fails = {**{f'nexar_{k}': r['failed'] for k, r in nexar.items() if r['failed']},
         **({'dota': dota['failed']} if dota['failed'] else {}),
         **({'cap_dev_uncompared': dota['cap_dev_uncompared']} if dota['cap_dev_uncompared'] else {})}
lines += ['', f'Failures: {json.dumps(fails)[:2000] if fails else "none"}']
(L_REP / 'r7_phase1.md').write_text('\n'.join(lines) + '\n')
(L_REP / 'r7_phase1.json').write_text(json.dumps({'pass': ok, 'nexar': nexar, 'dota': dota, 'gb': sizes}, indent=2))
sync_dir(L_REP, REPORTS)
print((L_REP / 'r7_phase1.md').read_text())
assert ok, 'R7 Phase 1 FAILED -- paste REPORTS/r7_phase1.md to Claude'
print('->', REPORTS / 'r7_phase1.md')